# Q1

In [24]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression


In [25]:
df = pd.read_csv("USA_Housing.csv")

In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 6 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Avg. Area Income              5000 non-null   float64
 1   Avg. Area House Age           5000 non-null   float64
 2   Avg. Area Number of Rooms     5000 non-null   float64
 3   Avg. Area Number of Bedrooms  5000 non-null   float64
 4   Area Population               5000 non-null   float64
 5   Price                         5000 non-null   float64
dtypes: float64(6)
memory usage: 234.5 KB


In [27]:
df.head()

,Avg. Area Income,Avg. Area House Age,Avg. Area Number of Rooms,Avg. Area Number of Bedrooms,Area Population,Price
0,79545.45857,5.682861,7.009188,4.09,23086.80050,1.059034e+06
1,79248.64245,6.002900,6.730821,3.09,40173.07217,1.505891e+06
2,61287.06718,5.865890,8.512727,5.13,36882.15940,1.058988e+06
3,63345.24005,7.188236,5.586729,3.26,34310.24283,1.260617e+06
4,59982.19723,5.040555,7.839388,4.23,26354.10947,6.309435e+05


In [28]:
X = df.drop("Price", axis=1).values
y = df["Price"].values


In [29]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [30]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

r2_manual = []
betas = []

print("\n--- Manual Implementation (Normal Equation) ---")
for fold, (train_idx, test_idx) in enumerate(kf.split(X_scaled), 1):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Add bias (intercept column)
    X_train_bias = np.c_[np.ones(X_train.shape[0]), X_train]
    X_test_bias = np.c_[np.ones(X_test.shape[0]), X_test]

    # Normal Equation: β = (X^TX)^(-1)X^Ty
    beta = np.linalg.inv(X_train_bias.T @ X_train_bias) @ X_train_bias.T @ y_train
    betas.append(beta)

    # Predictions
    y_pred = X_test_bias @ beta

    # R² score
    r2 = r2_score(y_test, y_pred)
    r2_manual.append(r2)
    print(f"Fold {fold}: R² = {r2:.4f}")

# Best beta (highest R²)
best_beta_idx = np.argmax(r2_manual)
best_beta = betas[best_beta_idx]
print("\nBest β (from Normal Equation):", best_beta[:6], "...")


--- Manual Implementation (Normal Equation) ---
Fold 1: R² = 0.9180
Fold 2: R² = 0.9146
Fold 3: R² = 0.9116
Fold 4: R² = 0.9193
Fold 5: R² = 0.9244

Best β (from Normal Equation): [1.23161736e+06 2.30225051e+05 1.63956839e+05 1.21115120e+05
 7.83467170e+02 1.50662447e+05] ...


In [31]:
# Using scikit-learn

print("\n--- Scikit-learn Implementation ---")
r2_sklearn = []
model = LinearRegression()

for fold, (train_idx, test_idx) in enumerate(kf.split(X_scaled), 1):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    r2_sklearn.append(r2)
    print(f"Fold {fold}: R² = {r2:.4f}")



--- Scikit-learn Implementation ---
Fold 1: R² = 0.9180
Fold 2: R² = 0.9146
Fold 3: R² = 0.9116
Fold 4: R² = 0.9193
Fold 5: R² = 0.9244


In [32]:
# Final 70–30 Split with Best β

print("\n--- Final Train-Test (70-30) using Best β ---")

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42
)

# Add bias
X_train_bias = np.c_[np.ones(X_train.shape[0]), X_train]
X_test_bias = np.c_[np.ones(X_test.shape[0]), X_test]

# Use best β
y_pred_final = X_test_bias @ best_beta

# R² Score
final_r2 = r2_score(y_test, y_pred_final)
print(f"Final R² Score (on 30% test data) = {final_r2:.4f}")



--- Final Train-Test (70-30) using Best β ---
Final R² Score (on 30% test data) = 0.9147


# Q2

In [33]:
from sklearn.linear_model import SGDRegressor
df = pd.read_csv("USA_Housing.csv")
X = df.drop("Price", axis=1).values
y = df["Price"].values

In [34]:
# Split dataset (56% train, 14% validation, 30% test)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.44, random_state=42
)


In [35]:
# Now split temp into validation (14%) and test (30%)
val_size = 14 / (14 + 30)  # proportion of validation in remaining 44%
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=1 - val_size, random_state=42
)

In [36]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [37]:
# Train with different learning rates

learning_rates = [0.001, 0.01, 0.1, 1]
results = {}

for lr in learning_rates:
    model = SGDRegressor(
        learning_rate="constant", eta0=lr, max_iter=1000, tol=None, random_state=42
    )
    model.fit(X_train, y_train)
    
    # Predictions
    y_val_pred = model.predict(X_val)
    y_test_pred = model.predict(X_test)
    
    # R² scores
    r2_val = r2_score(y_val, y_val_pred)
    r2_test = r2_score(y_test, y_test_pred)
    
    results[lr] = (r2_val, r2_test)


In [ ]:
print("\n--- R² Scores for Different Learning Rates ---")
for lr, (r2_val, r2_test) in results.items():
    print(f"Learning Rate = {lr} | Validation R² = {r2_val:.4f} | Test R² = {r2_test:.4f}")

# Best learning rate based on validation score
best_lr = max(results, key=lambda k: results[k][0])
print(f"\nBest Learning Rate: {best_lr}")
print(f"Best Validation R²: {results[best_lr][0]:.4f}")
print(f"Test R² (with best model): {results[best_lr][1]:.4f}")


--- R² Scores for Different Learning Rates ---
Learning Rate = 0.001 | Validation R² = 0.9195 | Test R² = 0.9135
Learning Rate = 0.01 | Validation R² = 0.9161 | Test R² = 0.9105
Learning Rate = 0.1 | Validation R² = 0.9058 | Test R² = 0.8993
Learning Rate = 1 | Validation R² = -89112476993756.3906 | Test R² = -85051867046715.2500

Best Learning Rate: 0.001
Best Validation R²: 0.9195
Test R² (with best model): 0.9135


# Q3

In [39]:
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/autos/imports-85.data"
columns = ["symboling","normalized_losses","make","fuel_type","aspiration","num_doors","body_style",
           "drive_wheels","engine_location","wheel_base","length","width","height","curb_weight",
           "engine_type","num_cylinders","engine_size","fuel_system","bore","stroke",
           "compression_ratio","horsepower","peak_rpm","city_mpg","highway_mpg","price"]

df = pd.read_csv(url, names=columns)


In [ ]:
df.replace("?", np.nan, inplace=True)

df.dropna(subset=["price"], inplace=True)

In [ ]:

numeric_cols = ["normalized_losses","wheel_base","length","width","height","curb_weight",
                "engine_size","bore","stroke","compression_ratio","horsepower",
                "peak_rpm","city_mpg","highway_mpg","price"]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")


df.fillna(df.mean(numeric_only=True), inplace=True)

In [ ]:
#  Handle Categorical Variables

num_map = {"two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "eight": 8, "twelve": 12}
df["num_doors"] = df["num_doors"].map(num_map)
df["num_cylinders"] = df["num_cylinders"].map(num_map)

df = pd.get_dummies(df, columns=["body_style", "drive_wheels"], drop_first=True)


label_cols = ["make", "aspiration", "engine_location", "fuel_type"]
le = LabelEncoder()
for col in label_cols:
    df[col] = le.fit_transform(df[col])


df["fuel_system"] = df["fuel_system"].apply(lambda x: 1 if str(x).startswith("pfi") else 0)


df["engine_type"] = df["engine_type"].apply(lambda x: 1 if str(x).startswith("ohc") else 0)


In [ ]:
X = df.drop("price", axis=1)
y = df["price"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Pipeline: impute missing values -> scale -> regression
from sklearn.linear_model import LinearRegression

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
r2_no_pca=r2_score(y_test, y_pred)
print("R² Score:", r2_no_pca)


R² Score: 0.8739504259107642


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ---- Linear Regression with PCA ----
pipeline_pca = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),                  
    ("pca", PCA(n_components=5)),                  
    ("model", LinearRegression())
])

pipeline_pca.fit(X_train, y_train)
y_pred_pca = pipeline_pca.predict(X_test)
r2_pca = r2_score(y_test, y_pred_pca)

print("R² Score with PCA:", r2_pca)

R² Score with PCA: 0.7866423975561807


In [51]:
# Compare

if r2_pca > r2_no_pca:
    print("✅ PCA improved test performance!")
    print(f"Best Accuracy: {r2_pca:.4f}")
else:
    print("❌ PCA did not improve test performance.")
    print(f"Best Accuracy: {r2_no_pca:.4f}")

❌ PCA did not improve test performance.
Best Accuracy: 0.8740
